In [1]:
# architecture: str = "convnet"
architecture: str = "scatnet"

In [ ]:
# load a trained model
from pathlib import Path

import torch as tc

from convnet import ConvNet
from scatnet import ScatNet

device = tc.device("cuda")

SHAPE = (1, 128, 128)
match architecture:
    case "convnet":
        model = ConvNet(shape=SHAPE).to(device)
    case "scatnet":
        model = ScatNet(shape=SHAPE).to(device)
    case other:
        raise ValueError(f"Unknown model {other}")

model_state_path = Path("./weights", f"{model.name}.pt")
model.load_state_dict(tc.load(model_state_path, weights_only=True))

RuntimeError: Error(s) in loading state_dict for ScatNet:
	Missing key(s) in state_dict: "classifier.1.weight", "classifier.1.bias", "classifier.3.weight", "classifier.3.bias", "classifier.5.weight", "classifier.5.bias". 
	Unexpected key(s) in state_dict: "classifier.2.weight", "classifier.2.bias". 

In [ ]:
import os

from torch.utils.data import DataLoader
import numpy as np

from datasets import CatsAndDogs

dataset = CatsAndDogs(Path(os.environ["CATS_AND_DOGS"]))
print(dataset.imgs[-10:])
print(dataset.class_to_idx)
index_to_class = {i: x[1] for i,x in enumerate(dataset.imgs)}

# could also use specific notable examples
class_0_examples = index_to_class[index_to_class == 0]
class_1_examples = index_to_class[index_to_class == 1]

# use dedicated loader for each class
# class_0_loader = DataLoader(dataset, sampler=class_0_examples)
# class_1_loader = DataLoader(dataset, sampler=class_1_examples)

SAMPLES = [0, 1000, 15000]
target_sample_loader = DataLoader(dataset, sampler=SAMPLES, batch_size=10)
images, labels = target_sample_loader
images, labels = images.to(device), labels.to(device)

In [ ]:
# XAI 1: LIME
from captum.attr import Lime
import matplotlib.pyplot as plt
import seaborn as sns

tc.manual_seed(0) # for perturbation determinism
np.random.seed(10)
lime = Lime(model)

# Compute attributions
attributions = lime.attribute(images, target=labels, n_samples=5000, perturbations_per_eval=200, show_progress=True)
# Convert into numpy for visualization
attributions_np = attributions.mean(dim=1).detach().cpu().numpy()  # [10, 3, 248, 496]

# Visualize attributions
fig, axes = plt.subplots(ncols=2, nrows=5, figsize=(13, 15))
axes = axes.flatten()
for i in range(10):
    sns.heatmap(images[i].mean(dim=0).detach().cpu().numpy(), ax=axes[i], cmap='gray', alpha=1, cbar=False)
    sns.heatmap(attributions_np[i], ax=axes[i], cmap='seismic', alpha=0.3, vmax=np.abs(attributions_np[i]).max(), vmin=-np.abs(attributions_np[i]).max(), cbar=True)
    axes[i].set_title(f'LIME attribution for class {labels[i].item()}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
##### XAI: Saliency
from captum.attr import Saliency

np.random.seed(1234)
idx_subjs = np.column_stack((np.random.randint(0, 200, size=4), np.random.randint(200, 298, size=4))).ravel()
input_img = torch.tensor(X_test[idx_subjs], requires_grad=True).float().unsqueeze(1).to(DEVICE)  # [10, 1, 248, 496]
img_label = torch.tensor(y_test[idx_subjs]).long().to(DEVICE)

# Define Saliency
xai = Saliency(model)

attributions = []

for i in tqdm(range(len(idx_subjs)), desc='Computing XAI attributions: '):
    # Compute attributions
    attributions.append(xai.attribute(input_img[i].unsqueeze(0), target=img_label[i], abs=False))
    attributions[i] = torch.squeeze(attributions[i]).detach().cpu().numpy()
attributions_np = gaussian_filter(np.stack(attributions), sigma=2.0)
#attributions_np[attributions_np <= 0.09] = 0.0

# Visualize attributions
fig, axes = plt.subplots(ncols=2, nrows=4, figsize=(13, 15))
axes = axes.flatten()

for i in range(8):
    sns.heatmap(input_img[i].mean(dim=0).detach().cpu().numpy(), ax=axes[i], cmap='gray', alpha=1, cbar=False)
    sns.heatmap(attributions_np[i], ax=axes[i], cmap='seismic', alpha=0.3, cbar=True, vmax=np.abs(attributions_np[i]).max(), vmin=-np.abs(attributions_np[i]).max())
    axes[i].set_title(f'Saliency attribution for class {img_label[i].item()}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
##### XAI: Input x Gradient
from captum.attr import InputXGradient

np.random.seed(1234)
idx_subjs = np.column_stack((np.random.randint(0, 200, size=4), np.random.randint(200, 298, size=4))).ravel()
input_img = torch.tensor(X_test[idx_subjs], requires_grad=True).float().unsqueeze(1).to(DEVICE)  # [10, 1, 248, 496]
img_label = torch.tensor(y_test[idx_subjs]).long().to(DEVICE)

# Define Input x Gradient
xai = InputXGradient(model)

attributions = []

for i in tqdm(range(len(idx_subjs)), desc='Computing XAI attributions: '):
    # Compute attributions
    attributions.append(xai.attribute(input_img[i].unsqueeze(0), target=img_label[i]))
    attributions[i] = torch.squeeze(attributions[i]).detach().cpu().numpy()
attributions_np = gaussian_filter(np.stack(attributions), sigma=2.0)
#attributions_np[attributions_np <= 0.09] = 0.0

# Visualize attributions
fig, axes = plt.subplots(ncols=2, nrows=4, figsize=(13, 15))
axes = axes.flatten()

for i in range(8):
    sns.heatmap(input_img[i].mean(dim=0).detach().cpu().numpy(), ax=axes[i], cmap='gray', alpha=1, cbar=False)
    sns.heatmap(attributions_np[i], ax=axes[i], cmap='seismic', alpha=0.3, cbar=True, vmax=np.abs(attributions_np[i]).max(), vmin=-np.abs(attributions_np[i]).max())
    axes[i].set_title(f'Input x Gradient attribution for class {img_label[i].item()}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
##### XAI: Guided Backpropagation
from captum.attr import GuidedBackprop

np.random.seed(1234)
idx_subjs = np.column_stack((np.random.randint(0, 200, size=4), np.random.randint(200, 298, size=4))).ravel()
input_img = torch.tensor(X_test[idx_subjs], requires_grad=True).float().unsqueeze(1).to(DEVICE)  # [10, 1, 248, 496]
img_label = torch.tensor(y_test[idx_subjs]).long().to(DEVICE)

# Define Guided Backpropagation
xai = GuidedBackprop(model)

attributions = []

for i in tqdm(range(len(idx_subjs)), desc='Computing XAI attributions: '):
    # Compute attributions
    attributions.append(xai.attribute(input_img[i].unsqueeze(0), target=img_label[i]))
    attributions[i] = torch.squeeze(attributions[i]).detach().cpu().numpy()
attributions_np = gaussian_filter(np.stack(attributions), sigma=2.0)
#attributions_np[attributions_np <= 0.09] = 0.0

# Visualize attributions
fig, axes = plt.subplots(ncols=2, nrows=4, figsize=(13, 15))
axes = axes.flatten()

for i in range(8):
    sns.heatmap(input_img[i].mean(dim=0).detach().cpu().numpy(), ax=axes[i], cmap='gray', alpha=1, cbar=False)
    sns.heatmap(attributions_np[i], ax=axes[i], cmap='seismic', alpha=0.3, cbar=True, vmax=np.abs(attributions_np[i]).max(), vmin=-np.abs(attributions_np[i]).max())
    axes[i].set_title(f'Guided Backpropagation attribution for class {img_label[i].item()}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()